# 01 — Model demonstration and controller masking (claim C-M, motivation)
Trajectories for every scenario; the masking picture (T pinned at setpoint, the fault moves into Qc and Tc);
open-loop multistability (why this reactor needs control); control-sensor offset vs monitoring drift.

In [ ]:
import json, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
NB_DIR = Path.cwd().resolve()
assert NB_DIR.name == "notebooks" and NB_DIR.parent.name == "system_III", f"run from system_III/notebooks (cwd {NB_DIR})"
ROOT = NB_DIR.parent
sys.path.insert(0, str(ROOT / "src"))
import importlib, system_iii
from system_iii import physics as ph, simulator as sim, features as fe, data as da, scenarios as scn
from system_iii import sensors as sen, information as info, config as cfg, rules, metrics as wm
for _m in (ph, sim, fe, da, scn, sen, info, cfg, rules, wm):
    importlib.reload(_m)
R, FIGS, DATA = ROOT / "results", ROOT / "figures", ROOT / "data"
for _d in (R, FIGS, DATA):
    _d.mkdir(exist_ok=True)
P = ph.PLANT
T0 = time.time()

In [ ]:
names = scn.CLOSED + scn.OPEN
out = {}
for i, k in enumerate(names):
    s = scn.SCENARIOS[k]
    out[k] = sim.simulate(np.full(20, s.alpha), np.full(20, s.beta), seed=100 + i, control=s.control,
                          drift_T_monitor=s.drift_T_monitor, oT=s.oT)
fig, axes = plt.subplots(4, 1, figsize=(10, 10), sharex=True, constrained_layout=True)
for k in ["Sc1_healthy", "Sc2_fouling", "Sc3_decay", "Sc4_combined", "Sc5_severe_fouling"]:
    o = out[k]
    for j, lab in enumerate(["C [mol/L]", "T [K]", "Tc [K]", "Qc [L/min]"]):
        axes[j].plot(o["t"], o["obs"][0, :, j], lw=0.8, label=k); axes[j].set_ylabel(lab)
axes[0].legend(fontsize=7, ncol=3); axes[-1].set_xlabel("t [min] (window)")
fig.savefig(FIGS / "01_trajectories.png", dpi=150); plt.show()
summ = pd.DataFrame({k: {"T_true": o["true"][:, :, 1].mean(), "T_logged": o["obs"][:, :, 1].mean(),
                         "C": o["obs"][:, :, 0].mean(), "Tc": o["obs"][:, :, 2].mean(), "Qc": o["obs"][:, :, 3].mean(),
                         "Qc_sat_frac": (o["qc_true"] >= P.Qcmax - 1e-6).mean()} for k, o in out.items()}).T
display(summ.round(3)); summ.to_csv(R / "01_scenario_means.csv")

## Masking: what a temperature-only monitor sees

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for k in ["Sc1_healthy", "Sc2_fouling", "Sc3_decay", "Sc4_combined"]:
    axes[0].hist(out[k]["obs"][:, :, 1].ravel(), bins=60, histtype="step", density=True, label=k)
    axes[1].hist(out[k]["obs"][:, :, 3].ravel(), bins=60, histtype="step", density=True, label=k)
axes[0].set_xlabel("logged T [K]"); axes[1].set_xlabel("logged Qc [L/min]"); axes[0].legend(fontsize=8)
axes[0].set_title("T: the controller hides the faults"); axes[1].set_title("Qc: where the fault information goes")
fig.savefig(FIGS / "01_masking.png", dpi=150); plt.show()

## Open-loop multistability

In [ ]:
Ts = np.linspace(295, 360, 600)
fig, ax = plt.subplots(figsize=(7, 4))
for a, b in [(1, 1), (1, 0.7), (0.75, 0.75), (0.7, 1)]:
    ax.plot(Ts, ph._f_open(P, Ts, a, b, P.Qc0, P.Ci, P.Ti, P.Tci, P.Q) / 1e6, label=f"alpha={a}, beta={b}")
    print((a, b), "open-loop steady states at nominal Qc:", [round(s["T"], 1) for s in ph.ss_open_all(P, a, b, P.Qc0)])
ax.axhline(0, color="k", lw=0.8); ax.set_xlabel("T [K]"); ax.set_ylabel("net heat [MJ/min]"); ax.legend(fontsize=8)
fig.savefig(FIGS / "01_open_loop_multiplicity.png", dpi=150); plt.show()

## Interpretation — to be written after the run
Write toward the storyline (`docs/system_III_plan.md` §1) and update the claims ledger (§2).